In [12]:
import openai
import requests
import json
import dotenv

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

PROMPT_TEMPLATE = """I have the following functions in my system.

`get_popular_movies()` - /movies 에서 인기 영화 목록을 가져옵니다. 인자 없음.
`get_movie_details(id)` - /movies/:id 에서 영화 상세 정보를 가져옵니다. 영화 ID를 인자로 받습니다.
`get_movie_credits(id)` - /movies/:id/credits 에서 출연진 및 제작진을 가져옵니다. 영화 ID를 인자로 받습니다.

Please answer with the name of function that you would like me to run.
If the function requires arguments, include them like: get_movie_details(550)

Please say nothing else, just the name of the function with the arguments.

Answer the following question:

{question}
"""

dotenv.load_dotenv()

client = openai.OpenAI()

In [13]:
def get_popular_movies():
    res = requests.get(f"{BASE_URL}/movies")
    return res.json()


def get_movie_details(movie_id):
    res = requests.get(f"{BASE_URL}/movies/{movie_id}")
    return res.json()


def get_movie_credits(movie_id):
    res = requests.get(f"{BASE_URL}/movies/{movie_id}/credits")
    return res.json()

def run_agent(user_input: str):
    # 1단계: LLM에게 어떤 함수를 호출할지 물어봄
    prompt = PROMPT_TEMPLATE.format(question=user_input)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    fn_call = response.choices[0].message.content.strip()
    print(f"[AI 선택] {fn_call}")

    # 2단계: 파싱 후 실제 함수 호출
    if fn_call.startswith("get_popular_movies"):
        result = get_popular_movies()
    elif fn_call.startswith("get_movie_details"):
        movie_id = fn_call.split("(")[1].rstrip(")")
        result = get_movie_details(movie_id)
    elif fn_call.startswith("get_movie_credits"):
        movie_id = fn_call.split("(")[1].rstrip(")")
        result = get_movie_credits(movie_id)
    else:
        print("알 수 없는 함수:", fn_call)
        return

    result_text = json.dumps(result, ensure_ascii=False, indent=2)
    # print(result_text)

    # 3단계: 함수 결과를 LLM에게 전달하여 최종 답변 생성 (토큰 아까우면 주석...)
    final_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": user_input},
            {
                "role": "system",
                "content": f"다음은 API 호출 결과입니다. 이 데이터를 바탕으로 사용자 질문에 한국어로 답변하세요.\n\n{result_text}",
            },
        ],
    )
    print(final_response.choices[0].message.content)




In [14]:
test_inputs = [
    "지금 인기 있는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘",
]

for q in test_inputs:
    print(f"\n{'='*60}")
    print(f"[질문] {q}")
    print("-" * 60)
    run_agent(q)


[질문] 지금 인기 있는 영화가 무엇인지 알려줘
------------------------------------------------------------
[AI 선택] get_popular_movies()
현재 인기 있는 영화 중 몇 가지를 소개해 드립니다:

1. **Your Heart Will Be Broken (Твоё сердце будет разбито)**
   - 개봉일: 2026년 3월 26일
   - 줄거리: 고등학생 폴리나는 새로운 학교에서 괴롭힘을 피하기 위해 주범 바르스와 거래를 합니다. 서로의 감정이 커지는 가운데 그들은 가족과 친구들로 인해 어려움에 직면합니다.
   - 평가: ★★★★☆ (6.6)

   ![Your Heart Will Be Broken](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)

2. **Avatar: Fire and Ash**
   - 개봉일: 2025년 12월 17일
   - 줄거리: 제이크 설리와 네이티리는 새로운 위협인 '재 사람들'과 싸우며 판도라의 미래를 지키기 위해 고군분투합니다.
   - 평가: ★★★★☆ (7.4)

   ![Avatar: Fire and Ash](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **Hoppers**
   - 개봉일: 2026년 3월 4일
   - 줄거리: 과학자들은 인간의 의식을 로봇 동물에 전이할 수 있는 방법을 발견하고, 동물과 의사소통을 하려는 한 여성이 그 기술을 활용하여 동물 세계의 신비를 탐구합니다.
   - 평가: ★★★★☆ (7.6)

   ![Hoppers](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)

4. **Project Hail Mary**
   - 개봉일: 2026년 3월 15일
   - 줄거리: 과학 교사인 라이랜드 